# Pandas Series 완전정복

주식 데이터 분석과 Pandas 활용을 위한 **Series 중심** 실습 노트북입니다.

## 목차

| 장 | 주제 | 핵심 키워드 |
|:--:|------|-------------|
| 1 | [Series란 무엇인가?](#1장-Series란-무엇인가) | `values`, `index`, `dtype` |
| 2 | [인덱스의 이해](#2장-인덱스의-이해) | 종목코드, `loc` |
| 3 | [Series 생성 방법](#3장-Series-생성-방법) | 리스트, 딕셔너리, numpy |
| 4 | [인덱싱과 슬라이싱](#4장-인덱싱과-슬라이싱) | `iloc`, `loc`, 불리언 |
| 5 | [Series 연산](#5장-Series-연산) | 스칼라·Series 간 연산 |
| 6 | [벡터화](#6장-벡터화-Vectorization) | `for` 대신 배열 연산 |
| 7 | [결측치](#7장-결측치-Missing-Values) | `isna`, `fillna`, `ffill` |
| 8 | [정렬과 순위](#8장-정렬과-순위) | `sort_values`, `rank` |
| 9 | [통계 함수](#9장-통계-함수) | `describe`, `mean`, `std` |
| 10 | [map()](#10장-map-완전정복) | 종목코드 → 종목명 |
| 11 | [apply()](#11장-apply) | 함수 적용 |
| 12 | [문자열 처리](#12장-문자열-처리-str) | `.str.contains` |
| 13 | [날짜 처리](#13장-날짜-처리) | `to_datetime`, `.dt` |
| 14 | [시계열 핵심](#14장-시계열-데이터-핵심) | `pct_change` |
| 15 | [rolling()](#15장-rolling--이동-평균) | 이동평균 MA |
| 16 | [groupby()](#16장-groupby-입문) | 업종별 집계 |
| 17 | [Series → DataFrame](#17장-Series--DataFrame) | `to_frame`, `concat` |
| 18 | [실전 프로젝트](#18장-실전-프로젝트) | FinanceDataReader |
| — | [종합 퀴즈](#-종합-퀴즈) | 복습 |

## 학습 목표
- Series가 무엇이고 DataFrame과 어떻게 다른지 이해한다
- `loc` / `iloc` 인덱싱을 구분해서 사용한다
- 결측치, 정렬, 통계, `map` / `apply`를 실무 맥락에서 다룬다
- 시계열(`pct_change`, `rolling`)과 `groupby` 기초를 익힌다

## 사용 환경
```bash
pip install pandas numpy finance-datareader
```

> 각 장의 **📝 실습** 셀은 직접 코드를 수정·실행해 보세요.

## 1장. Series란 무엇인가?

**Series** = 1차원 배열 + **인덱스(라벨)**

| 구분 | Series | DataFrame |
|------|--------|-----------|
| 차원 | 1차원 (한 열) | 2차원 (표) |
| 비유 | 종목 1개의 시계열 종가 | 여러 종목·여러 컬럼 표 |

주식 예시: `종가` 한 줄만 다룰 때 → Series, `종가·거래량·등락률` 함께 → DataFrame


In [ ]:
import pandas as pd
import numpy as np

# 리스트로 Series 생성 — 기본 인덱스는 0, 1, 2, ...
s = pd.Series([10, 20, 30])
print('=== Series 내용 ===')
print(s)
print('\n=== 주요 속성 ===')
print('값(values):', s.values)   # numpy 배열
print('인덱스(index):', s.index)  # 라벨
print('dtype:', s.dtype)
print('길이:', len(s))

### 📝 실습 1-1
`[100, 200, 300]` 거래량 Series `volume`을 만들고 `values`와 두 번째 값을 출력해 보세요.

In [ ]:
# 직접 작성 후 실행
volume = pd.Series([100, 200, 300])
print(volume.values)
print(volume[1])

## 2장. 인덱스의 이해

주식에서는 **종목코드**가 인덱스가 되는 경우가 많습니다.

- `s['005930']` → 라벨로 접근 = **`loc`**
- `s[0]` → 위치로 접근 = **`iloc`**


In [ ]:
# 종목코드를 인덱스로 지정
s = pd.Series([80000, 250000], index=['005930', '000660'])
print(s)
print()

# loc: 라벨(종목코드)로 접근
print('삼성전자 종가:', s['005930'])
print('loc 사용:', s.loc['000660'])

# 인덱스 이름만 변경 (값은 동일)
s_named = s.copy()
s_named.index = ['삼성전자', 'SK하이닉스']
print('\n인덱스 변경 후:')
print(s_named)

## 3장. Series 생성 방법

| 방법 | 예시 | 언제 쓰나 |
|------|------|-----------|
| 리스트 | `pd.Series([1,2,3])` | 순서 있는 값 |
| 딕셔너리 | `pd.Series({'A':1})` | **키=인덱스** 자동 지정 |
| numpy | `pd.Series(np.arange(10))` | 수치 계산 결과 |
| 기존 열 | `df['Close']` | DataFrame에서 추출 |


In [ ]:
# 1) 딕셔너리 — 키가 인덱스가 됨
price_dict = pd.Series({'삼성전자': 80000, 'SK하이닉스': 250000})
print('딕셔너리 생성:\n', price_dict, sep='')

# 2) numpy 배열
nums = pd.Series(np.arange(10))
print('\nnumpy 생성:\n', nums, sep='')

# 3) 등락률 리스트 (주식 실무 예시)
change_pct = pd.Series([1.2, -0.5, 3.1], index=['005930', '000660', '035420'])
print('\n등락률 Series:\n', change_pct, sep='')

## 4장. 인덱싱과 슬라이싱

| 방식 | 의미 | 예시 |
|------|------|------|
| `iloc` | **위치** 기반 | `s.iloc[0]` 첫 번째 |
| `loc` | **라벨** 기반 | `s.loc['005930']` |
| 슬라이싱 | 구간 선택 | `s[:2]`, `s.loc['005930':'000660']` |

⚠️ `loc` 슬라이싱은 **끝 라벨 포함**, `iloc`는 파이썬처럼 끝 제외


In [ ]:
s = pd.Series([80000, 250000, 180000], index=['005930', '000660', '035420'])

print('iloc[0] (첫 번째):', s.iloc[0])
print("loc['000660']:", s.loc['000660'])

# 위치 슬라이싱 — 0, 1번만 (2번 제외)
print('\niloc[:2]:\n', s.iloc[:2], sep='')

# 라벨 슬라이싱 — 000660 **포함**
print("\nloc['005930':'000660']:\n", s.loc['005930':'000660'], sep='')

# 불리언 인덱싱 — 조건 필터
print('\n8만원 이상:\n', s[s >= 80000], sep='')

## 5장. Series 연산

같은 인덱스끼리는 **자동 정렬·맞춤(align)** 후 연산됩니다.

- 스칼라 연산: 모든 원소에 적용 (`price * 1.1`)
- Series 간 연산: 인덱스가 같으면 원소별 계산


In [ ]:
price = pd.Series([80000, 250000], index=['삼성전자', 'SK하이닉스'])

# 10% 상승 가정
target = price * 1.1
print('목표가 (+10%):\n', target, sep='')

# 두 종목 합산 포트폴리오 가치
qty = pd.Series([10, 5], index=['삼성전자', 'SK하이닉스'])  # 보유 수량
value = price * qty
print('\n평가금액:\n', value, sep='')
print('총 평가금액:', value.sum())

## 6장. 벡터화 (Vectorization)

`for` 루프 대신 **배열 연산**을 쓰면 훨씬 빠르고 코드가 짧아집니다.

```python
# 느림
for i in range(len(price)):
    price[i] = price[i] + 1000

# 빠름 (벡터화)
price + 1000
```


In [ ]:
price = pd.Series([80000, 250000], index=['삼성전자', 'SK하이닉스'])

# 벡터 연산: 전 종목 +1000원
adjusted = price + 1000
print(adjusted)

# 비교 연산도 벡터화
print('\n20만원 이상?:', (price >= 200000).tolist())

# numpy 함수와 함께
print('로그 변환:', np.log10(price).round(2).tolist())

## 7장. 결측치 (Missing Values)

주식 데이터에서 결측은 휴장, 상장 전, 데이터 오류 등으로 발생합니다.

| 메서드 | 설명 |
|--------|------|
| `isna()` | 결측 여부 (True/False) |
| `notna()` | 값이 있는지 |
| `fillna(0)` | 결측을 0으로 채움 |
| `dropna()` | 결측 행 제거 |


In [ ]:
# None, np.nan 모두 결측으로 처리
s = pd.Series([1, 2, None, 4, np.nan], index=['월', '화', '수', '목', '금'])
print('원본:\n', s, sep='')
print('\n결측 위치:\n', s.isna(), sep='')
print('\n결측 개수:', s.isna().sum())

# 0으로 채우기
print('\nfillna(0):\n', s.fillna(0), sep='')

# 앞 값으로 채우기 (시계열에서 자주 사용)
print('\nffill:\n', s.ffill(), sep='')

## 8장. 정렬과 순위

- `sort_values()` : 값 기준 정렬
- `sort_index()` : 인덱스(종목코드) 기준 정렬
- `rank()` : 순위 (1위, 2위, …) — 상대 순위 비교에 유용


In [ ]:
# 등락률(%) 예시
change = pd.Series([1.2, -0.5, 3.1, 0.0], index=['삼성전자', 'SK하이닉스', 'NAVER', '카카오'])

print('등락률 내림차순:\n', change.sort_values(ascending=False), sep='')
print('\n순위 (높을수록 1위):\n', change.rank(ascending=False), sep='')

# 상위 2개만
print('\n상위 2종목:\n', change.sort_values(ascending=False).head(2), sep='')

## 9장. 통계 함수

한눈에 분포를 보려면 `describe()`가 편합니다.

| 함수 | 의미 |
|------|------|
| `mean()` | 평균 |
| `median()` | 중앙값 |
| `std()` | 표준편차 (변동성) |
| `min()` / `max()` | 최소/최대 |


In [ ]:
returns = pd.Series([1.2, -0.5, 3.1, 0.8, -1.0], name='등락률')

print('=== describe() 요약 ===')
print(returns.describe())

print('\n평균 등락률:', round(returns.mean(), 2), '%')
print('변동성(std):', round(returns.std(), 2))
print('최대 상승:', returns.max(), '%')

## 10장. map() 완전정복

**각 원소를 1:1로 변환**할 때 사용합니다. 주식에서 종목코드 → 종목명 변환이 대표적입니다.

- `map(딕셔너리)` : 키에 맞는 값으로 치환
- 매칭 안 되면 `NaN` 반환


In [ ]:
ticker = pd.Series(['005930', '000660', '035420'], name='종목코드')
name_dict = {
    '005930': '삼성전자',
    '000660': 'SK하이닉스',
    '035420': 'NAVER',
}

# 코드 → 이름
names = ticker.map(name_dict)
print('종목명:\n', names, sep='')

# 등락구분 코드 → 한글 (실무 패턴)
code_map = {'1': '상승', '2': '하락', '0': '보합'}
change_code = pd.Series(['1', '2', '1'])
print('\n등락구분:\n', change_code.map(code_map), sep='')

## 11장. apply()

**함수를 각 원소에 적용**합니다. `map`보다 유연하지만 보통 더 느립니다.

- 단순 치환 → `map` 우선
- 조건 분기·복잡한 계산 → `apply`


In [ ]:
price = pd.Series([80000, 250000, 52000], index=['삼성전자', 'SK하이닉스', '현대차'])

# lambda: 간단한 변환
print('2배:\n', price.apply(lambda x: x * 2), sep='')

# 조건 함수
def price_label(p):
    if p >= 200000:
        return '고가주'
    if p >= 100000:
        return '중가주'
    return '저가주'

print('\n가격대 분류:\n', price.apply(price_label), sep='')

## 12장. 문자열 처리 (`.str`)

Series에 **`.str`** 를 붙이면 문자열 메서드를 벡터화해서 쓸 수 있습니다.

종목명 필터, 접두사 검색 등에 자주 사용합니다.


In [ ]:
names = pd.Series(['삼성전자', 'SK하이닉스', '삼성SDI', 'LG전자'])

print("'전자' 포함?:\n", names.str.contains('전자'), sep='')
print("\n'삼성'으로 시작?:\n", names.str.startswith('삼성'), sep='')
print('\n대문자 변환:\n', names.str.upper(), sep='')

# 필터 결과로 Series 추출
print('\n전자 관련 종목:\n', names[names.str.contains('전자')], sep='')

## 13장. 날짜 처리

`pd.to_datetime()`으로 변환하면 **`.dt`** 로 연·월·요일 등을 꺼낼 수 있습니다.

주식 시계열에서 월별 집계, 요일 패턴 분석에 활용합니다.


In [ ]:
dates = pd.to_datetime(pd.Series(['2026-01-02', '2026-02-01', '2026-03-03']))
print('날짜 Series:\n', dates, sep='')
print('\n월:', dates.dt.month.tolist())
print('요일(0=월):', dates.dt.dayofweek.tolist())
print('분기:', dates.dt.quarter.tolist())

# 날짜 차이 (Timedelta)
print('\n첫날 대비 일수 차:', (dates - dates.iloc[0]).dt.days.tolist())

## 14장. 시계열 데이터 핵심

**`pct_change()`** = 전일 대비 수익률(%) 계산. 첫 행은 비교 대상이 없어 `NaN`입니다.

```python
수익률 = (오늘 - 어제) / 어제 * 100
```


In [ ]:
# 4일간 종가 시뮬레이션
close = pd.Series([100, 110, 105, 120], index=pd.date_range('2026-01-01', periods=4))
print('종가:\n', close, sep='')

# 일간 수익률 (소수: 0.1 = 10%)
daily_return = close.pct_change()
print('\n일간 수익률:\n', (daily_return * 100).round(2), sep='')

# 누적 수익률
cum_return = (1 + daily_return).cumprod() - 1
print('\n누적 수익률:\n', (cum_return * 100).round(2), sep='')

## 15장. rolling() — 이동 평균

**최근 N일** 윈도우로 통계를 계산합니다. 주식에서 **이동평균선(MA)** 에 해당합니다.

- `rolling(5).mean()` → 5일 이동평균
- 초기 N-1개는 윈도우가 채워지지 않아 `NaN`


In [ ]:
close = pd.Series([100, 110, 105, 120, 115, 130, 125])

ma2 = close.rolling(2).mean()   # 2일 이동평균
ma3 = close.rolling(3).mean()   # 3일 이동평균

print('종가:\n', close, sep='')
print('\nMA(2):\n', ma2.round(1), sep='')
print('\nMA(3):\n', ma3.round(1), sep='')

# 이동 표준편차 — 변동성 지표
print('\n3일 변동성(std):\n', close.rolling(3).std().round(2), sep='')

## 16장. groupby() 입문

DataFrame에서 **그룹별로 Series를 집계**할 때 사용합니다.

예: 업종별 평균 등락률, 섹터별 종목 수


In [ ]:
df = pd.DataFrame({
    '업종': ['반도체', '반도체', '인터넷', '인터넷', '금융'],
    '종목': ['삼성전자', 'SK하이닉스', 'NAVER', '카카오', 'KB금융'],
    '수익률': [1.2, 2.5, -0.5, 3.1, 0.8],
})

# 업종별 평균 수익률 → 결과는 Series
sector_avg = df.groupby('업종')['수익률'].mean()
print('업종별 평균 수익률:\n', sector_avg, sep='')

# 여러 통계 한 번에
print('\n업종별 요약:\n', df.groupby('업종')['수익률'].agg(['mean', 'max', 'count']), sep='')

## 17장. Series → DataFrame

Series는 `to_frame()`으로 DataFrame 열 1개로 바꿀 수 있습니다.

여러 Series를 `pd.concat`하면 표를 만듭니다.


In [ ]:
close = pd.Series([80000, 250000], index=['005930', '000660'], name='종가')
volume = pd.Series([10_000_000, 3_000_000], index=['005930', '000660'], name='거래량')

# Series → 1열 DataFrame
print(close.to_frame())

# 여러 Series 합치기
stock_df = pd.concat([close, volume], axis=1)
print('\n종목 표:\n', stock_df, sep='')

## 18장. 실전 프로젝트

FinanceDataReader로 삼성전자 데이터를 받아 **종가 Series**로 시계열 분석을 해 봅니다.

### 📝 종합 실습
1. `Close` Series를 만든다
2. `pct_change()`로 일간 수익률을 구한다
3. `rolling(5).mean()`으로 5일 이동평균을 구한다
4. 최근 5일 평균 수익률과 변동성(`std`)을 출력한다


In [ ]:
import FinanceDataReader as fdr

# 삼성전자 최근 3개월 데이터 (네트워크 필요)
df = fdr.DataReader('005930', '2025-12-01', '2026-03-01')
print('DataFrame 미리보기:')
print(df.head(3))

# ① DataFrame의 'Close' 열 → Series
close = df['Close']
print('\n종가 Series (타입):', type(close))
print(close.tail(3))

# ② 일간 수익률 (%)
daily_ret = close.pct_change() * 100

# ③ 5일 이동평균
ma5 = close.rolling(5).mean()

# ④ 최근 5거래일 요약
print('\n=== 최근 5일 요약 ===')
summary = pd.DataFrame({
    '종가': close,
    '수익률%': daily_ret.round(2),
    'MA5': ma5.round(0),
})
print(summary.tail(5))
print(f"\n최근 5일 평균 수익률: {daily_ret.tail(5).mean():.2f}%")
print(f"최근 5일 변동성(std): {daily_ret.tail(5).std():.2f}%")

---

## 📝 종합 퀴즈

1. `loc['B']`와 `iloc[1]` — 같은 값을 가리키는 조건은?
2. 결측치가 있을 때 `mean()`은 기본적으로 결측을 어떻게 처리할까?
3. 종목코드→종목명 변환에 `map` vs `apply` 중 추천은?
4. `rolling(5).mean()` 앞 4개가 NaN인 이유는?